<a href="https://colab.research.google.com/github/Statofthe7/ZeroShotMultilingualSentimentAnalysisSOVvsSVO/blob/main/MultilingualSentimentAnalysis_updated.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update # Up to date list of packages
# !apt-get install -y mecab libmecab-dev # Mecab and its libraries for Fugashi

In [ ]:
!pip install datasets pandas transformers torch sentencepiece
# !pip install fugashi[unidic-lite]

!python -m spacy download fr_core_news_sm
!python -m spacy download ja_core_news_sm

In [ ]:
# from fugashi import Tagger
from transformers import AutoTokenizer
from transformers import pipeline
import warnings
import pandas as pd
from datasets import load_dataset, concatenate_datasets
from tqdm import tqdm # for progress bar
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from scipy import stats
import spacy
import numpy as np


In [ ]:
# This script loads the Allociné (French) and Japanese Generic Sentiment datasets
# It also calculates average and maximum "length" of text reviews in both datasets
# For French, length is count of words in each review
# For Japanese, lenght is count of morphemes or sub-word tokens

# mDeBERTa-v3 is a multilingual version of DeBERTa (Decoding-enhanced BERT with distangled attention) and is a medium sized pre-trained model
# As part of fine-tuning on mnli and xnli datasets, it has learned to reason about a premise's entailment
# and contradiction to a hypothesis (NLI-Natural Language Inference) across different languages
tokenizer = AutoTokenizer.from_pretrained("MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")

# Load the French Dataset (Allociné Movie Reviews)
print("Loading French data...")
fr_data = load_dataset("allocine", split='test')

# Load the Japanese Sentiment Dataset
print("Loading Japanese data...")
jp_data = load_dataset("mteb/JapaneseSentimentClassification", split='test')

# Setup Japanese Tagger to breakup text into space separated (owakati) morphemes
#tagger = Tagger('-Owakati')

# Get token length in Japanese
def get_japanese_token_length(text):

    # Using fugashi to parse text to morphemes separated by spaces and split it to form a list
    # tokens = tagger.parse(text).split()

    # Using the transformer model's tokenizer (SentencePiece) to parse japanese text to sub-word tokens (similar to morphemes)
    tokens = tokenizer.encode(text)

    return len(tokens)

def get_stats(dataset, text_column, lang):
    df = pd.DataFrame(dataset)

    if lang == 'fr':
        # French: Split by spaces
        df['length'] = df[text_column].apply(lambda x: len(x.split()))
    else:
        # Japanese: Split by morphemes or sub-word tokens
        df['length'] = df[text_column].apply(get_japanese_token_length)

    return df['length'].mean(), df['length'].max()

fr_mean, fr_max = get_stats(fr_data, 'review', 'fr')
jp_mean, jp_max = get_stats(jp_data, 'text', 'jp')

print(f"\n--- Statistical Results ---")
print(f"French  - Avg Words per Review: {fr_mean:.2f} (Max: {fr_max})")
print(f"Japanese - Avg Morphs per Review: {jp_mean:.2f} (Max: {jp_max})")

In [ ]:
# This script initializes the classifier to perform zero shot classification

# Zero-Shot Pipeline is a NLP technique to perform classification on categories the pre-trained language model hasn't been trained on

classifier = pipeline("zero-shot-classification",
                      model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli",
                      device=-1) # Change to 0 if you have a GPU/CUDA

# Run Tests
def test_sentiment(text, lang_name):
    candidate_labels = ["positive", "negative"]

    # We use a 'hypothesis template' to guide the model's classification logic
    # This works across languages because the model is multilingual
    hypothesis_template = "The sentiment of this text is {}.";

    result = classifier(text, candidate_labels, hypothesis_template=hypothesis_template)

    print(f"\n--- {lang_name} Analysis ---")
    print(f"Text: {text}")
    print(f"Top Label: {result['labels'][0]} ({result['scores'][0]:.2%})")

fr_text = "Le film était absolument magnifique, bien que la fin soit un peu triste."
jp_text = "このカメラの画質は素晴らしいですが、バッテリーの持ちが悪いです。" # "Good quality, but bad battery"

test_sentiment(fr_text, "French")
test_sentiment(jp_text, "Japanese")

In [ ]:
# This script runs sentiment analysis on both the datasets and writes results into two csv files

# Ensures we have a fair 50/50 split of Positive and Negative samples
def get_balanced_data(dataset, num_per_class=50):
    neg = dataset.filter(lambda x: x['label'] == 0).shuffle(seed=42).select(range(num_per_class))
    pos = dataset.filter(lambda x: x['label'] == 1).shuffle(seed=42).select(range(num_per_class))

    print(f"  Retrieved {len(pos)} positive samples and {len(neg)} negative samples (target: {num_per_class} per class).")

    return concatenate_datasets([neg, pos]).shuffle(seed=42)


balanced_fr_data = get_balanced_data(fr_data)
balanced_jp_data = get_balanced_data(jp_data)

def run_experiment(dataset, text_col, label_col, lang_name, num_samples=100):
    results = []
    print(f"Analyzing {lang_name}...")

    df_sample = pd.DataFrame(dataset).sample(num_samples, random_state=42)

    for _, row in tqdm(df_sample.iterrows(), total=num_samples):
        raw_text = row[text_col]

        # Calculate Length based on language
        if lang_name == "Japanese":
           # processed_text = tagger.parse(raw_text)
            processed_text = raw_text
            token_length = len(tokenizer.encode(raw_text))
        else:
            processed_text = raw_text
            token_length = len(raw_text.split())

        true_label_idx = row[label_col]
        true_label = "positive" if true_label_idx == 1 else "negative"

        # Run AI prediction
        output = classifier(processed_text, ["positive", "negative"],
                                    hypothesis_template="The sentiment is {}.")

        pred_label = output['labels'][0]
        confidence = output['scores'][0]

        results.append({
            "raw_text": raw_text,
            "processed_text": processed_text,
            "token_length": token_length,
            "true_label": true_label,
            "pred_label": pred_label,
            "correct": true_label == pred_label,
            "confidence": confidence
        })

    return pd.DataFrame(results)

# Execute
df_fr = run_experiment(balanced_fr_data, 'review', 'label', "French")
df_jp = run_experiment(balanced_jp_data, 'text', 'label', "Japanese")

# Save to CSV
df_fr.to_csv("french_results.csv", index=False)
df_jp.to_csv("japanese_results.csv", index=False)

print("\nAccuracy French:", df_fr['correct'].mean())
print("Accuracy Japanese:", df_jp['correct'].mean())

In [ ]:
#This script charts confusion matrices for French and Japanese sentiment analysis

def plot_research_results(csv_file, lang_name):
    df = pd.read_csv(csv_file)

    # Create the Confusion Matrix
    # This shows: True Positives, False Positives, True Negatives, False Negatives
    cm = confusion_matrix(df['true_label'], df['pred_label'], labels=["positive", "negative"])

    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=["Positive", "Negative"],
                yticklabels=["Positive", "Negative"])

    plt.title(f'Confusion Matrix: {lang_name} Sentiment Analysis')
    plt.ylabel('Actual Label')
    plt.xlabel('Predicted Label')
    plt.show()

# Run for both
plot_research_results("french_results.csv", "French")
plot_research_results("japanese_results.csv", "Japanese")

In [ ]:
# This script shows a comparison chart for accuracy of the results
acc_fr = pd.read_csv("french_results.csv")['correct'].mean()
acc_jp = pd.read_csv("japanese_results.csv")['correct'].mean()

plt.bar(['French', 'Japanese'], [acc_fr, acc_jp], color=['#3498db', '#e74c3c'])
plt.ylabel('Accuracy Score')
plt.title('Zero-Shot Sentiment Accuracy: French vs. Japanese')
plt.ylim(0, 1) # Set scale from 0 to 100%
plt.show()



In [ ]:
# This script is to visually see if token length affects model accuracy differently in
# French (SVO) vs. Japanese (SOV).

def plot_length_vs_accuracy(csv_file, lang_name):
    df = pd.read_csv(csv_file)

    # Create length "bins" (e.g., 0-10 words, 10-20, etc.)
    # This makes the trend much easier to see than a scatter plot
    df['length_bin'] = pd.cut(df['token_length'],bins=[0, 20, 40, 60, 100, 200, float('inf')],
                              labels=['(0-20)', '(20-40)', '(40-60)', '(60-100)', '(100-200)', '(200+)'])

    plt.figure(figsize=(10, 6))

    # Plotting mean accuracy for each bin
    sns.barplot(data=df, x='length_bin', y='correct', hue='length_bin', palette='magma', errorbar=None, legend=False)

    plt.title(f'Accuracy by Model Token Count: {lang_name}')
    plt.ylabel('Accuracy Rate')
    plt.xlabel('Number of Sub-word Tokens')
    plt.ylim(0, 1.1)

    # Add a horizontal line for the overall average
    plt.axhline(df['correct'].mean(), color='red', linestyle='--', label='Avg Accuracy')
    plt.legend()
    plt.show()

# Run for both results files
plot_length_vs_accuracy("french_results.csv", "French")
plot_length_vs_accuracy("japanese_results.csv", "Japanese")

In [ ]:
# This script shows statistical summary table with mean accuracy, correlation r
# and significance p for French and Japanese

def generate_statistical_summary(french_csv, japanese_csv):
    # Load your results
    df_fr = pd.read_csv(french_csv)
    df_jp = pd.read_csv(japanese_csv)

    summary_data = []

    for df, lang in [(df_fr, "French"), (df_jp, "Japanese")]:
        # Mean Accuracy
        mean_acc = df['correct'].mean()

        # Point-Biserial Correlation (r and p)
        # We correlate 'token_length' (continuous) with 'correct' (binary)
        # r (Correlation Coefficient) ranges from -1 to +1.
        # r - A value close to +1 indicates strong positive correlation. This means as length of text review increases, the likelihood of predicting sentiment also incresases
        # r - A value close to -1 indicates negative correlation. This means as length of text review increases, the likelihood of predicting sentiment decreases
        # r - A value close to 0 indicates little to no linear correlation
        # p (P-Value) shows if the observed correlation between token length and accuracy of predicted sentiment is statistically significant or likely due to random chance
        # p - If less than the conventional significance level of 0.05, the correlation is considered statistically significant. This means there is a strong correlation that likely exists in the larger population
        # p - If greater than 0.05, the correlation is generally not considered statistically significant. This means the weak correlation might be random variation

        r_val, p_val = stats.pointbiserialr(df['correct'], df['token_length'])

        summary_data.append({
            "Language": lang,
            "Mean Accuracy": f"{mean_acc:.2%}",
            "Correlation (r)": f"{r_val:.3f}",
            "P-Value": f"{p_val:.4f}",
            "Significance": "Significant" if p_val < 0.05 else "Not Significant"
        })

    # Create the final table
    summary_table = pd.DataFrame(summary_data)
    return summary_table

# Execute
results_table = generate_statistical_summary("french_results.csv", "japanese_results.csv")
print(results_table)

# Save for review
results_table.to_csv("statistical_summary.csv", index=False)

## Part 2: Full Multi-Model Analysis (Paper Tables & Statistics)

The cells below reproduce all tables and statistics reported in the paper using the full 500-sample per language datasets. LLM predictions (Gemini 3 Pro, Claude Opus 4.5, GPT-5.2) were collected manually via each model's consumer app interface on February 12, 2026, using batch prompting (see Methods). The results are stored in the Excel files (`french_results1000.xlsx`, `japanese_results1000.xlsx`) which must be uploaded to Colab before running these cells.

> **Note on LLM reproducibility:** Because the LLM labels were collected through consumer app interfaces using batch prompting rather than API calls, they cannot be automatically re-run from code. The raw labels are preserved in the Excel files as the primary data record. See the discussion cell below for options.


In [ ]:
# Install additional dependency for reading Excel files
!pip install openpyxl --quiet


In [ ]:
# ── Load full 1000-sample results from Excel files ──────────────────────────
# Upload french_results1000.xlsx and japanese_results1000.xlsx to Colab before running.
# If running locally, adjust paths accordingly.

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.metrics import confusion_matrix

def load_results(filepath):
    """Load model results from Excel, returning a clean DataFrame."""
    df = pd.read_excel(filepath, engine='openpyxl')
    # Drop trailing empty columns
    df = df.dropna(axis=1, how='all')
    # Standardise column names
    df.columns = [c.strip() if isinstance(c, str) else c for c in df.columns]
    # Drop trailing empty rows
    df = df.dropna(subset=['true_label'])
    # Ensure boolean correct columns
    for col in ['mDeBERTa-v3Correct', 'GeminiCorrect', 'ClaudeCorrect', 'ChatGPTCorrect']:
        if col in df.columns:
            df[col] = df[col].astype(bool)
    return df

df_fr = load_results('sample_data/french_results1000.xlsx')
df_jp = load_results('sample_data/japanese_results1000.xlsx')

print(f"French N={len(df_fr)}, Japanese N={len(df_jp)}")
print(f"French label balance: {df_fr['true_label'].value_counts().to_dict()}")
print(f"Japanese label balance: {df_jp['true_label'].value_counts().to_dict()}")
df_fr.head(3)


In [ ]:
# ── TABLE 1: Overall accuracy and cross-lingual gap ─────────────────────────
# Chi-square tests use Yates' continuity correction (scipy default: correction=True).

model_cols = {
    'mDeBERTa-v3': 'mDeBERTa-v3Correct',
    'Gemini':       'GeminiCorrect',
    'Claude':       'ClaudeCorrect',
    'ChatGPT':      'ChatGPTCorrect',
}

rows = []
for model_name, col in model_cols.items():
    jp_c = df_jp[col].sum(); jp_n = len(df_jp)
    fr_c = df_fr[col].sum(); fr_n = len(df_fr)
    jp_acc = jp_c / jp_n
    fr_acc = fr_c / fr_n
    gap = jp_acc - fr_acc
    contingency = [[jp_c, jp_n - jp_c], [fr_c, fr_n - fr_c]]
    chi2, p, _, _ = stats.chi2_contingency(contingency, correction=True)
    sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
    rows.append({
        'Model': model_name,
        'Japanese Accuracy': f'{jp_acc:.1%}',
        'French Accuracy':   f'{fr_acc:.1%}',
        'Gap':               f'{gap:.1%}',
        'χ²':                f'{chi2:.2f}',
        'p-value':           f'{p:.3f}{sig}',
    })

table1 = pd.DataFrame(rows)
print("Table 1. Overall accuracy and cross-lingual gap by model.")
print("Note: χ² tests use Yates' continuity correction. *p<0.05, **p<0.01, ***p<0.001\n")
print(table1.to_string(index=False))


In [ ]:
# ── FIGURE 1: Accuracy by model and language ────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 5))
models = list(model_cols.keys())
x = np.arange(len(models))
width = 0.35

jp_accs = [df_jp[col].mean() for col in model_cols.values()]
fr_accs = [df_fr[col].mean() for col in model_cols.values()]

bars_jp = ax.bar(x - width/2, jp_accs, width, label='Japanese', color='#2E86AB', alpha=0.9)
bars_fr = ax.bar(x + width/2, fr_accs, width, label='French',   color='#E84855', alpha=0.9)

# Annotate bars
for bar in bars_jp + bars_fr:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x); ax.set_xticklabels(models)
ax.set_ylabel('Accuracy'); ax.set_ylim(0.6, 1.05)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_title('Figure 1. Overall Accuracy by Model and Language')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figure1_accuracy.png', dpi=150)
plt.show()
print(f"Saved: figure1_accuracy.png")


In [ ]:
# ── TABLE 2: Point-biserial correlations (document length vs. accuracy) ─────
# Bins use exclusive lower bound: (lo, hi] i.e., lo < token_length <= hi

rows2 = []
for model_name, col in model_cols.items():
    for lang, df in [('Japanese', df_jp), ('French', df_fr)]:
        lengths = df['token_length'].values
        correct = df[col].astype(int).values
        r, p = stats.pointbiserialr(lengths, correct)
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else ''))
        rows2.append({
            'Model': model_name, 'Language': lang,
            'r': f'{r:+.3f}', 'p': f'{p:.3f}{sig}',
        })

table2 = pd.DataFrame(rows2)
# Pivot for paper format
table2_wide = table2.pivot(index='Model', columns='Language', values=['r','p'])
table2_wide.columns = ['French r','Japanese r','French p','Japanese p']
table2_wide = table2_wide[['Japanese r','Japanese p','French r','French p']]
table2_wide = table2_wide.loc[list(model_cols.keys())]
print("Table 2. Point-biserial correlation between token length and accuracy.")
print("Note: *p<0.05, **p<0.01, ***p<0.001\n")
print(table2_wide.to_string())


In [ ]:
# ── TABLE 3: Accuracy by token-length bin (mDeBERTa and ChatGPT) ────────────
# Bin convention: exclusive lower bound, inclusive upper: (lo, hi]

bin_edges  = [0, 20, 40, 60, 100, 200, float('inf')]
bin_labels = ['0–20','20–40','40–60','60–100','100–200','200+']

for df in [df_fr, df_jp]:
    df['length_bin'] = pd.cut(df['token_length'], bins=bin_edges, labels=bin_labels, right=True)

def bin_accuracy(df, col):
    grp = df.groupby('length_bin', observed=False)[col].agg(['sum','count'])
    grp['acc'] = grp['sum'] / grp['count']
    return grp

rows3 = []
for label in bin_labels:
    row = {'Token Bin': label}
    for model_name, col in [('mDeBERTa', 'mDeBERTa-v3Correct'), ('ChatGPT', 'ChatGPTCorrect')]:
        for lang, df in [('JP', df_jp), ('FR', df_fr)]:
            sub = df[df['length_bin'] == label]
            acc = sub[col].mean() if len(sub) > 0 else float('nan')
            n   = len(sub)
            row[f'{model_name} {lang}'] = f'{acc:.1%} (n={n})'
    rows3.append(row)

table3 = pd.DataFrame(rows3)
print("Table 3. Accuracy by token-length bin for mDeBERTa-v3 and ChatGPT.")
print()
print(table3.to_string(index=False))


In [ ]:
# ── FIGURE 2: Length-bin accuracy — mDeBERTa vs ChatGPT ────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
colors = {'mDeBERTa-v3Correct': '#E84855', 'ChatGPTCorrect': '#2E86AB'}
labels_map = {'mDeBERTa-v3Correct': 'mDeBERTa-v3', 'ChatGPTCorrect': 'ChatGPT'}

for ax, (lang_name, df) in zip(axes, [('Japanese', df_jp), ('French', df_fr)]):
    for col, color in colors.items():
        grp = df.groupby('length_bin', observed=False)[col].mean()
        ax.plot(bin_labels, grp.values, marker='o', label=labels_map[col], color=color, linewidth=2)
    ax.set_title(f'Figure 2. {lang_name}: Accuracy by Token-Length Bin')
    ax.set_xlabel('Token-Length Bin'); ax.set_ylabel('Accuracy')
    ax.set_ylim(0.4, 1.05)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.legend(); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('figure2_length_bins.png', dpi=150)
plt.show()
print("Saved: figure2_length_bins.png")


In [ ]:
# ── Supplementary: label composition of long French reviews ─────────────────
# Reported in Discussion (Issue 9): 200+ token bin is 68% positive

long_fr = df_fr[df_fr['length_bin'] == '200+']
label_counts = long_fr['true_label'].value_counts()
pct_pos = label_counts.get('positive', 0) / len(long_fr)
print(f"French 200+ token bin: n={len(long_fr)}, positive={label_counts.get('positive',0)} ({pct_pos:.0%}), negative={label_counts.get('negative',0)}")

# mDeBERTa error breakdown in 200+ bin
errors = long_fr[long_fr['mDeBERTa-v3Correct'] == False]
print(f"mDeBERTa errors in 200+ bin: {len(errors)} total")
print(f"  Positive reviews misclassified: {(errors['true_label']=='positive').sum()}")
print(f"  Negative reviews misclassified: {(errors['true_label']=='negative').sum()}")


In [ ]:
# ── SUPPLEMENTARY: LLM classification via API (for replication only) ─────────

# !pip install anthropic openai google-generativeai --quiet
# import anthropic, openai, google.generativeai as genai

ANTHROPIC_API_KEY = "YOUR_KEY_HERE"
OPENAI_API_KEY    = "YOUR_KEY_HERE"
GEMINI_API_KEY    = "YOUR_KEY_HERE"

PROMPT_TEMPLATE = (
    "Classify the sentiment of the following review as either 'positive' or 'negative'. "
    "Respond with a single word only.\n\nReview: {text}\n\nSentiment:"
)

def classify_claude(text, client):
    msg = client.messages.create(
        model="claude-opus-4-5",  # Use the version available at time of replication
        max_tokens=5,
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}]
    )
    response = msg.content[0].text.strip().lower()
    return 'positive' if 'positive' in response else 'negative'

def classify_chatgpt(text, client):
    resp = client.chat.completions.create(
        model="gpt-5.2",
        max_tokens=5,
        messages=[{"role": "user", "content": PROMPT_TEMPLATE.format(text=text)}]
    )
    response = resp.choices[0].message.content.strip().lower()
    return 'positive' if 'positive' in response else 'negative'

def classify_gemini(text, model):
    response = model.generate_content(PROMPT_TEMPLATE.format(text=text))
    result = response.text.strip().lower()
    return 'positive' if 'positive' in result else 'negative'

# Example usage (uncomment to run):
# client_anthropic = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
# client_openai    = openai.OpenAI(api_key=OPENAI_API_KEY)
# genai.configure(api_key=GEMINI_API_KEY)
# gemini_model = genai.GenerativeModel('gemini-1.5-pro')  # Use version available at replication time
#
# from tqdm import tqdm
# df_fr['Claude_replicated'] = [classify_claude(t, client_anthropic) for t in tqdm(df_fr['raw_text'])]
# df_fr['Claude_replicated_correct'] = df_fr['Claude_replicated'] == df_fr['true_label']
# print(f"Replicated Claude accuracy (French): {df_fr['Claude_replicated_correct'].mean():.1%}")

print("API replication cell loaded. Set API keys and uncomment code to run.")
print("Note: results will differ from the paper due to model version and inference mode differences.")
